In [16]:
from anemone.inversion import *
from anemone.forwards import Forward, logify_forward
from anemone.system import Loop, Receiver, Waveform, FilterChain, ButterworthFilter
from anemone.experiment import compute_relstd

%matplotlib qt

logForward = logify_forward(Forward)

times = torch.logspace(-5,-2, 32)
fwrTD = logForward.example(times=times, tolerance=1e-5)

rho = torch.tensor([20, 80, 40])
thk = torch.tensor([30,40.])
mod_true = LogResistivity.few_layers(rho, thk)


dbdt_true = 10**fwrTD(rho, thk)
d_relerr = compute_relstd(fwrTD.times, dbdt_true, current=10.)
d_err = torch.log10(1. + d_relerr)
d_obs = dbdt_true + dbdt_true * torch.randn_like(dbdt_true) * d_relerr

d_obs = torch.abs(d_obs).log10()

res_start = LogResistivity.smooth(torch.ones(31, dtype=torch.double)*50, torch.cumsum(torch.logspace(0,1.5,30), 0))

inv = Tikhonov(d_obs, d_err, res_start, fwrTD)
inv.data_misfit_acceptance_threshold = 0.9
res_mod = inv.invert(iter_cap=10)        

if __name__ == '__main__':
    import matplotlib.pyplot as plt

    fig,axs = plt.subplots(1,2)
    
    ax = res_start.plot(axs=axs[:1], color='black', linestyle='--')
    ax = res_mod.plot(axs=axs[:1])
    mod_true.plot(axs=axs[:1], color='red', label='True model')

    d_pred = fwrTD(*res_mod.mod)
    axs[1].loglog(fwrTD.times, 10**d_obs, 'o', label='Observed')
    axs[1].loglog(fwrTD.times, 10**d_pred, '-', label='Predicted')
    plt.show()

001	misfit:	4.80
002	misfit:	1.91
003	misfit:	1.20
004	misfit:	1.02
005	misfit:	0.97
006	misfit:	0.94
007	misfit:	0.93
008	misfit:	0.91
009	misfit:	0.89
misfit is below 0.9, stopping inversion


In [17]:
from anemone.inversion import *
from anemone.forwards import Forward, logify_forward
from anemone.system import Loop, Receiver, Waveform, FilterChain, ButterworthFilter
from anemone.experiment import compute_relstd
import matplotlib.pyplot as plt
import numpy as np




times = torch.logspace(-5,-2, 32)
tx = Loop.square(40)
rx = Receiver()
fwrTD = Forward.mpa(tx, rx, times, waveform=Waveform.example(), tolerance=1e-6)

In [18]:

rho=  torch.tensor([5.000e+02,5.000e+02,5.000e+02,8.000e+02,8.000e+02,5.000e+02,5.000e+02,5.000e+02,5.000e+02,2.000e+02,2.000e+02,5.000e+02,5.000e+02,5.000e+02,5.000e+02,5.000e+02,5.000e+02,8.000e+02,8.000e+02,5.000e+02])
phi = torch.tensor([2.000e+02,2.000e+02,2.000e+02,2.000e+02,2.000e+02,5.000e+02,5.000e+02,5.000e+02,5.000e+02,8.000e+02,8.000e+02,8.000e+02,8.000e+02,8.000e+02,8.000e+02,8.000e+02,8.000e+02,8.000e+02,8.000e+02,8.000e+02])/1e3
tau = torch.tensor([1.000e-04,1.000e-04,1.000e-04,2.000e-04,2.000e-04,1.000e-04,1.000e-04,9.000e-04,9.000e-04,9.000e-04,1.000e-04,1.000e-04,1.000e-04,1.000e-04,1.000e-04,1.000e-04,1.000e-04,1.000e-04,1.000e-04,1.000e-04])
C =   torch.tensor([6.000e-01,6.000e-01,4.000e-01,4.000e-01,6.000e-01,6.000e-01,6.000e-01,6.000e-01,6.000e-01,3.000e-01,3.000e-01,3.000e-01,3.000e-01,6.000e-01,6.000e-01,6.000e-01,6.000e-01,6.000e-01,6.000e-01,6.000e-01])
thick=torch.tensor([1.000e+00, 1.177e+00, 1.386e+00, 1.632e+00, 1.922e+00, 2.263e+00, 2.665e+00, 3.138e+00, 3.695e+00, 4.351e+00, 5.124e+00, 6.034e+00, 7.105e+00, 8.366e+00, 9.851e+00, 1.160e+01, 1.365e+01, 1.608e+01, 1.893e+01])

cmod = MPA.smooth([rho, phi, tau, C], z_bounds=torch.cumsum(thick,0))

dbdt = fwrTD(*cmod.mod)


dbdt_true = fwrTD(*cmod.mod)
d_relerr = compute_relstd(fwrTD.times, dbdt_true, current=10.)
d_err = torch.log10(1. + d_relerr)
d_obs = dbdt_true + dbdt_true * torch.randn_like(dbdt_true) * d_relerr

# d_obs = torch.abs(d_obs)#.log10()

z = torch.logspace(-1, 2.1, 30)

res_start = MPA.smooth([torch.ones(len(z)+1)*x.mean() for x in [rho, phi, tau, C]], z_bounds=z)

inv = Tikhonov(d_obs, dbdt_true*d_relerr, res_start, fwrTD)
inv.setup_matrices()
jac = inv.jacobian(res_start.vec)
inv.invert(iter_cap=10)

001	misfit:	23.78
002	misfit:	23.77
misfit change is less than 0.001


<dolphinv.models.BaseModel.smooth.<locals>.SmoothModel at 0x195f9293c50>

In [19]:
jac.shape

torch.Size([32, 124])

In [20]:
plt.figure()
plt.imshow(np.log10(np.abs(jac.numpy())), aspect='auto', cmap='viridis')

In [21]:
from anemone.physics import *

def calc_perturbation(sigma0, sigma_pert, pert_div, thickness, omega, wave_length, mu=c.mu0, epsilon=c.epsilon0, propmat_answer=None):
    pad = 2
    return_propmat = False
    if propmat_answer is None:
        propmat = calc_prop_matrix(sigma0, thickness, omega, wave_length, mu=mu, epsilon=epsilon, pad=pad)
        propmat = torch.roll(propmat, -pad//2, dims=2)

        answer = torch.zeros((2,)+propmat.shape[2:], dtype=torch.complex128, device=thickness.device)
        
        answer[0, :] = 1.
        for i in range(answer.shape[1]-2, -1, -1):
            answer[:, i] = torch.einsum('ij...,j...->i...', propmat[:,:,i], answer[:, i+1])

        for i in range(1, propmat.shape[2]):
            propmat[:,:,i] = torch.einsum('ij...,jk...->ik...', propmat[:,:, i-1], propmat[:,:, i])

        

        answer = torch.roll(answer, -1, dims=1)
        answer[0,-1:] = 1.
        answer[1,-1:] = 0.
        return_propmat = True

    else:
        propmat,answer = propmat_answer
    rte = answer[1, 0] / answer[0, 0]

        
    dsigma = sigma0.clone()
    dsigma[::2] = sigma_pert[::2]
    dpropmat = calc_prop_matrix(dsigma, thickness, omega, wave_length, mu=mu, epsilon=epsilon, pad=pad)
    dpropmat = torch.roll(dpropmat, -pad//2, dims=2)

    drte = torch.einsum('ij...,j...->i...', dpropmat, answer)
    drte = torch.roll(drte, -1, dims=1)
    drte[0,-1] = 1.
    drte[1,-1] = 0.
    drte = torch.einsum('ij...,j...->i...', dpropmat, drte)
    drte = torch.roll(drte, -1, dims=1)
    drte[0,-1] = 1.
    drte[1,-1] = 0.
    drte = torch.einsum('ij...,j...->i...', propmat, drte)

    dsigma = sigma0.clone()
    dsigma[1::2] = sigma_pert[1::2]
    dpropmat = calc_prop_matrix(dsigma, thickness, omega, wave_length, mu=mu, epsilon=epsilon, pad=pad)
    dpropmat = torch.roll(dpropmat, -pad//2, dims=2)
    
    drte2 = torch.einsum('ij...,j...->i...', dpropmat, answer)
    drte2 = torch.roll(drte2, -1, dims=1)
    drte2[0,-1] = 1.
    drte2[1,-1] = 0.
    drte2 = torch.einsum('ij...,j...->i...', dpropmat, drte2)
    drte2 = torch.roll(drte2, -1, dims=1)
    drte2[0,-1] = 1.
    drte2[1,-1] = 0.
    drte2 = torch.einsum('ij...,j...->i...', propmat,  drte2)
    drte[:, 1::2] = drte2[:, 1::2]

    
    mid = slice(None, len(sigma0))
    if return_propmat:
        return torch.nan_to_num((drte[1, mid]/drte[0, mid] - rte))/pert_div, propmat, answer
    else:
        return torch.nan_to_num((drte[1, mid]/drte[0, mid] - rte))/pert_div


In [22]:
from anemone.transforms import invFourier
from anemone.physics import mpa2colecole


class MPA(Forward):
    sigma0 = None

    @property
    def sigma(self):
        rho, m_new, tau_cc, C = mpa2colecole(self.rho, self.phi_max, self.tau_phi, self.c)
        return 1./colecole(rho, m_new, tau_cc, C, self.omega)
    
    @staticmethod
    def colecole2mpa(rho, m, tau, C):
        from anemone.physics import colecole2mpa # not used much, so imported here to keep things clean.
        rho, phi_max, tau_phi, C = colecole2mpa(rho, m, tau, C)
        return rho, phi_max, tau_phi, C


    def jacobian(self):
        def frq_integrand(f):
            self.omega = c.twopi*f[None,:,None]
            def wvlngth_integrand(wave_length):
                """ Hz from horizontal electric dipole. """
                lmb = wave_length[None, None, :]          
                self.sfield_rte = torch.exp(-lmb*self.sfield_dz)*lmb / c.fourpi
                if self.sigma0 is None:
                    self.sigma0 = self.sigma*1.
                    self.div_pert = 0.02*self.rho
                    self.rho += self.div_pert
                    intg, self.propmat,self.answer = calc_perturbation(self.sigma0, self.sigma, self.div_pert, self.thick, self.omega, lmb, mu=c.mu0, epsilon=c.epsilon0)
                    self.rho -= self.div_pert
                else:
                    intg = calc_perturbation(self.sigma0, self.sigma, self.div_pert, self.thick, self.omega, lmb, mu=c.mu0, epsilon=c.epsilon0, propmat_answer=(self.propmat,self.answer))
                return torch.squeeze(intg*self.sfield_rte)
            Hr = torch.einsum('...i,i->...', wvlngth_integrand(self.lmb.squeeze()), self.geomz)
            if self.filterfunc is not None:
                Hr *= self.filterfunc.omega_filter(self.omega.squeeze())
            if self.waveform is not None:
                Hr /= -1j*self.omega.squeeze()
            return Hr.squeeze()
        tmp_cbft = invFourier(frq_integrand, device=self.device)
        Hrho = torch.einsum('ij,...j->...i', self.t2t, tmp_cbft.f1(self.omega1.squeeze())*self.omega1.squeeze())

        self.div_pert = 0.02*self.phi_max
        self.phi_max += self.div_pert
        Hphi = torch.einsum('ij,...j->...i', self.t2t, tmp_cbft.f1(self.omega1.squeeze())*self.omega1.squeeze())
        self.phi_max -= self.div_pert

        self.div_pert = 0.02*self.tau_phi
        self.tau_phi += self.div_pert
        Htau = torch.einsum('ij,...j->...i', self.t2t, tmp_cbft.f1(self.omega1.squeeze())*self.omega1.squeeze())
        self.tau_phi -= self.div_pert

        self.div_pert = 0.02*self.c
        self.c += self.div_pert
        Hc = torch.einsum('ij,...j->...i', self.t2t, tmp_cbft.f1(self.omega1.squeeze())*self.omega1.squeeze())
        self.c -= self.div_pert

        Ht = torch.vstack([Hrho, Hphi, Htau, Hc]).T
        self.sigma0 = None
        return Ht.real
    
        
    def __call__(self, rho, phi_max, tau_phi, C, thickness, jac=False):
        self.rho     =     rho.to(self.device)[..., None,None]
        self.phi_max = phi_max.to(self.device)[..., None,None]
        self.tau_phi = tau_phi.to(self.device)[..., None,None]
        self.c       =       C.to(self.device)[..., None,None]
        if rho.ndim > thickness.ndim:
            thickness = thickness[:, None]
        self.thick = ((thickness+0j)[..., None, None]).to(self.device)

        if jac:
            G = self.jacobian()
            return G*c.mu0# / rho[:, None]**2
        else:            
            return self.forward()*c.mu0    


In [23]:
fastfwr = MPA(fwrTD.source, fwrTD.receiver, fwrTD.times, waveform=fwrTD.waveform, tolerance=fwrTD.tolerance)
dbdt = fastfwr(*res_start.mod)
jac2 = fastfwr(*res_start.mod, jac=True)

In [24]:
%%timeit
jac2 = fastfwr(*res_start.mod, jac=True)

221 ms ± 17 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [25]:
%%timeit
inv.jacobian(res_start.vec)

1.18 s ± 22.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [26]:
jac2.shape


torch.Size([32, 124])

In [27]:
vlim = max(np.abs(jac.numpy()).max(), np.abs(jac2.numpy()).max())
fig,axs=plt.subplots(3,1)
pcm=axs[0].pcolor(jac.numpy(), cmap='seismic', vmin=-vlim, vmax=vlim)
plt.colorbar(pcm, ax=axs[0])
pcm=axs[1].pcolor(jac2.numpy(), cmap='seismic', vmin=-vlim, vmax=vlim)
plt.colorbar(pcm, ax=axs[1])
pcm=axs[2].pcolor((jac-jac2).numpy(), cmap='seismic', vmin=-vlim/1e11, vmax=vlim/1e11)
plt.colorbar(pcm, ax=axs[2])

In [28]:
vlim = max(np.abs(jac.numpy()).max(), np.abs(jac2.numpy()).max())
fig,axs=plt.subplots(3,1)
pcm=axs[0].pcolor(np.log10(np.abs(jac.numpy())), cmap='viridis')
plt.colorbar(pcm, ax=axs[0])
pcm=axs[1].pcolor(np.log10(np.abs(jac.numpy())), cmap='viridis')
plt.colorbar(pcm, ax=axs[1])
pcm=axs[2].pcolor((jac-jac2).numpy(), cmap='seismic', vmin=-vlim/1e11, vmax=vlim/1e11)
plt.colorbar(pcm, ax=axs[2])